In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import copy

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root = '/content/drive/MyDrive/PaanaAI', transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)
class_names = ['attacking midfielder', 'central midfielder', 'centre back', 'defensive midfielder', 'left back', 'left winger', 'right back', 'right winger', 'striker']

In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    # Block 1: (3, 128, 64) -> pool -> (32, 64, 32)
    self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
    self.pool1 = nn.MaxPool2d(2, 2)
    self.bn1   = nn.BatchNorm2d(32)

    # Block 2: (32, 64, 32) -> pool -> (64, 32, 16)
    self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
    self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
    self.pool2 = nn.MaxPool2d(2, 2)
    self.bn2   = nn.BatchNorm2d(64)

    # Block 3: (64, 32, 16) -> pool -> (128, 16, 8)
    self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
    self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
    self.pool3 = nn.MaxPool2d(2, 2)
    self.bn3   = nn.BatchNorm2d(128)

    # 128 * 16 * 8 = 16384
    self.dropout = nn.Dropout(0.5)
    self.fc1 = nn.Linear(128 * 16 * 8, 512)
    self.fc2 = nn.Linear(512, 128)
    self.fc3 = nn.Linear(128, 9)

  def forward(self, x):
    x = self.pool1(F.relu(self.bn1(self.conv2(F.relu(self.conv1(x))))))
    x = self.pool2(F.relu(self.bn2(self.conv4(F.relu(self.conv3(x))))))
    x = self.pool3(F.relu(self.bn3(self.conv6(F.relu(self.conv5(x))))))
    x = x.view(x.size(0), -1)
    x = self.dropout(F.relu(self.fc1(x)))
    x = self.dropout(F.relu(self.fc2(x)))
    x = self.fc3(x)
    return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [ ]:
train_loss, val_loss = [], []
train_acc, val_acc   = [], []

# early stopping state
best_val_loss    = float('inf')
patience         = 10
patience_counter = 0
best_model_state = None

for epoch in range(30):
    # Training
    model.train()
    running_loss          = 0.0
    train_correct, train_total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted  = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total   += labels.size(0)

    scheduler.step()

    # Validation
    model.eval()
    running_val_loss      = 0.0
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs         = model(images)
            running_val_loss += criterion(outputs, labels).item()
            _, predicted     = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total   += labels.size(0)

    # Record metrics
    epoch_train_loss = running_loss     / len(train_loader)
    epoch_val_loss   = running_val_loss / len(val_loader)
    epoch_train_acc  = 100 * train_correct / train_total
    epoch_val_acc    = 100 * val_correct   / val_total

    train_loss.append(epoch_train_loss)
    val_loss.append(epoch_val_loss)
    train_acc.append(epoch_train_acc)
    val_acc.append(epoch_val_acc)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.1f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.1f}%")

    # Early stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss    = epoch_val_loss
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())   # save best weights
    else:
        patience_counter += 1
        print(f"           No improvement — patience {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}.")
            break

# Restore the best weights after training ends
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Best model weights restored.")

In [ ]:
epochs_ran = range(1, len(train_loss) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(epochs_ran, train_loss, label='Train Loss',  marker='o')
ax1.plot(epochs_ran, val_loss,   label='Val Loss',    marker='o')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(epochs_ran, train_acc, label='Train Accuracy', marker='o')
ax2.plot(epochs_ran, val_acc,   label='Val Accuracy',   marker='o')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Detailed breakdown per class
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()